# Fine-tuning do Gemma 4 E2B para triagem de incidentes

Este notebook prepara os dados, ajusta o Gemma 4 E2B com LoRA e avalia o resultado na identificação de incidentes de infraestrutura.

**Logs usados:** Traefik, Docker/Swarm, Kubernetes, PostgreSQL, Authentik e Prometheus.

**Formato de saída:** JSON com `severidade`, `componente_afetado`, `causa_raiz` e `runbook`.

## Antes de executar

- Rode as células na ordem, de cima para baixo.
- Use o kernel **Gemma4 (Py 3.12)**.
- Deixe na mesma pasta os arquivos `gerar_dataset.py`, `triagem.py`, `preparar_reais.py` e a pasta `runbooks/`.
- A pasta `logs_raw/` é opcional. Ela contém os logs reais usados na avaliação.


## 1. Preparar os dados

Gera os arquivos de treino e teste em `data/`. Se a pasta `logs_raw/` existir, também prepara `reais/teste_real.jsonl` para a avaliação com logs do ambiente.


In [ ]:
!rm -rf data
!python gerar_dataset.py
!test -d logs_raw && python preparar_reais.py || echo "(sem logs_raw/: pulando a preparação dos logs reais)"


## 2. Login no Hugging Face (opcional)

Faça login nesta etapa apenas se for publicar o adaptador no Hugging Face. Também é possível pular agora e autenticar na hora do upload.


In [ ]:
from huggingface_hub import notebook_login

notebook_login()


## 3. Imports


In [ ]:
# O Unsloth precisa ser importado antes de transformers e trl.
from unsloth import FastModel
from unsloth.chat_templates import train_on_responses_only

import gc
import json
from functools import partial
from pathlib import Path

import torch
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

import triagem as tri


## 4. Configurações

Preencha `SEU_USUARIO` com seu nome de usuário do Hugging Face antes de publicar o modelo.

In [ ]:
MODEL_ID = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 2048  # Sequências maiores que o limite serão removidas do treino.
EPOCHS = 2
N_BASELINE = 40
OUTPUT_DIR = "outputs"
ADAPTER_DIR = "gemma4-e2b-incident-triage-v6"

SEU_USUARIO = "SEU_USUARIO"  # Informe seu usuário do Hugging Face.
HF_REPO = f"{SEU_USUARIO}/gemma4-e2b-incident-triage"


def status_gpu(etapa: str) -> None:
    livre, total = torch.cuda.mem_get_info()
    print(
        f"[{etapa}] VRAM livre {livre / 1e9:.2f} GB de {total / 1e9:.2f} GB "
        f"| PyTorch usando {torch.cuda.memory_allocated() / 1e9:.2f} GB"
    )


assert torch.cuda.is_available(), "CUDA não disponível."
status_gpu("início")

## 5. Carregar o modelo

O modelo é carregado em 4-bit. A célula mostra o consumo de memória da GPU ao terminar o carregamento.


In [ ]:
BASE_KWARGS = {
    "model_name": MODEL_ID,
    "max_seq_length": MAX_SEQ_LENGTH,
    "load_in_4bit": True,
    "full_finetuning": False,
    "text_only": True,
}

try:
    model, tokenizer = FastModel.from_pretrained(
        **BASE_KWARGS,
        device_map={"": 0},
        offload_embedding=True,
    )
except (ValueError, torch.OutOfMemoryError) as erro:
    print(
        f"Não foi possível carregar apenas na GPU ({type(erro).__name__}); "
        "tentando com offload_layers='auto'."
    )
    gc.collect()
    torch.cuda.empty_cache()
    model, tokenizer = FastModel.from_pretrained(
        **BASE_KWARGS,
        offload_layers="auto",
    )

status_gpu("modelo carregado")


## 6. Configurar o LoRA

O ajuste será feito por meio de adaptadores LoRA. Apenas uma parte pequena dos parâmetros do modelo será treinada.


In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    random_state=42,
)


## 7. Preparar o dataset

Aplica o template de conversa do tokenizer e remove exemplos de treino que ultrapassariam o limite de sequência. Assim, a resposta em JSON não é cortada durante o treinamento.


In [ ]:
def aplicar_template(lote):
    textos = [
        tokenizer.apply_chat_template(
            conversa,
            tokenize=False,
            add_generation_prompt=False,
        )
        for conversa in lote["conversations"]
    ]
    bos = getattr(tokenizer, "bos_token", None)
    if bos:
        textos = [texto.removeprefix(bos) for texto in textos]
    return {"text": textos}


dados = load_dataset(
    "json",
    data_files={"treino": "data/treino.jsonl", "teste": "data/teste.jsonl"},
)
treino = dados["treino"].map(
    aplicar_template,
    batched=True,
    remove_columns=dados["treino"].column_names,
)

# Remove exemplos que poderiam ter a resposta cortada pelo limite de tokens.
antes = len(treino)
treino = treino.filter(
    lambda ex: len(
        tokenizer(text=ex["text"], add_special_tokens=False)["input_ids"]
    ) < MAX_SEQ_LENGTH - 16
)
teste = dados["teste"]

print(
    f"Treino: {len(treino)} exemplos "
    f"(descartados por tamanho: {antes - len(treino)}) | Teste: {len(teste)} exemplos"
)


## 8. Funções de inferência e avaliação

As funções vêm de `triagem.py`. A função `triagem()` também valida o runbook retornado: caminhos que não estejam no catálogo são substituídos por `None`.


In [ ]:
analisar_logs = partial(tri.analisar_logs, model, tokenizer)
triagem = partial(tri.triagem, model, tokenizer)
avaliar = partial(tri.avaliar, model, tokenizer)

print(f"Runbooks disponíveis: {len(tri.RUNBOOKS_VALIDOS)}")


## 9. Avaliação antes do treino (baseline)

Mede o desempenho do modelo base antes do fine-tuning. A avaliação usa uma amostra do conjunto sintético e, quando disponível, também os logs reais do ambiente.


In [ ]:
# Avaliação do modelo base, antes do fine-tuning.
baseline = avaliar(teste, N_BASELINE, "MODELO BASE (antes do treino)")

# Repete a avaliação com logs reais, se o arquivo estiver disponível.
if Path("reais/teste_real.jsonl").exists():
    reais = load_dataset("json", data_files="reais/teste_real.jsonl")["train"]
    baseline_real = avaliar(reais, len(reais), "MODELO BASE em logs reais")

# Exibe uma resposta sem tratamento para inspeção.
_, bruto_base = analisar_logs(
    teste[0]["conversations"][0]["content"],
    prompt_pronto=True,
)
print("\nResposta do modelo base:\n", bruto_base[:800])


## 10. Treinar o modelo

O treinamento usa duas épocas por padrão. Para um teste mais rápido, altere `EPOCHS` na seção de configurações.

A loss é calculada apenas sobre a resposta do modelo, não sobre o prompt nem sobre o conteúdo dos logs.


In [ ]:
def detectar_marcadores(tok):
    """Identifica os marcadores que delimitam a entrada e a resposta no template."""
    msg = [{"role": "user", "content": "@@U@@"}]
    sem = tok.apply_chat_template(msg, tokenize=False)
    com = tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    inicio_user = sem.split("@@U@@")[0].removeprefix(
        getattr(tok, "bos_token", "") or ""
    )
    return inicio_user, com[len(sem):]


trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=treino,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=EPOCHS,
        warmup_steps=10,
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        bf16=True,
        seed=42,
        output_dir=OUTPUT_DIR,
        report_to="none",
    ),
)

inicio_user, inicio_modelo = detectar_marcadores(tokenizer)
print("Marcadores:", repr(inicio_user), repr(inicio_modelo))
trainer = train_on_responses_only(
    trainer,
    instruction_part=inicio_user,
    response_part=inicio_modelo,
)

status_gpu("antes do treino")
stats = trainer.train()
print(
    f"Treino: {stats.metrics['train_runtime'] / 60:.1f} min | "
    f"pico VRAM {torch.cuda.max_memory_reserved() / 1e9:.2f} GB"
)


## 11. Testar com um log

Carrega o adaptador salvo e executa a triagem de um exemplo. Substitua o conteúdo de `exemplo` por um log do seu ambiente para testar outros casos.


In [ ]:
import json
from functools import partial
from pathlib import Path

from unsloth import FastModel
import triagem as tri

# Carrega o adaptador salvo na etapa de treinamento.
model, tokenizer = FastModel.from_pretrained(
    model_name="gemma4-e2b-incident-triage-v6",
    max_seq_length=2048,
    load_in_4bit=True,
    text_only=True,
    device_map={"": 0},
    offload_embedding=True,
)
FastModel.for_inference(model)

# Prepara as funções de triagem para o modelo ajustado.
triagem = partial(tri.triagem, model, tokenizer)
avaliar = partial(tri.avaliar, model, tokenizer)

exemplo = """traefik-1  | 2026-10-31T12:01:10Z INF Starting provider *docker.Provider
api-1      | INFO:     172.18.0.4:51234 - \"GET /health HTTP/1.1\" 200 OK
api-1      | java.sql.SQLTransientConnectionException: HikariPool-1 - Connection is not available, request timed out after 30000ms (total=20, active=20, idle=0, waiting=37)
traefik-1  | 201.17.4.9 - - [31/Oct/2026:12:01:41 +0000] \"POST /v1/checkout HTTP/2.0\" 504 15 \"-\" \"okhttp/4.12.0\" 912 \"api@docker\" \"http://172.18.0.4:8080\" 30004ms"""

resultado = triagem(exemplo)

if resultado:
    print(json.dumps(resultado, indent=2, ensure_ascii=False))
else:
    print("O modelo não retornou um JSON válido. Verifique o log de entrada.")

## 12. Avaliação final com dados sintéticos

Compara as previsões do modelo ajustado com os rótulos do conjunto de teste sintético. Observe principalmente a identificação do runbook, o grounding e a taxa de previsões inventadas.


In [ ]:
teste = load_dataset("json", data_files={"teste": "data/teste.jsonl"})["teste"]

final = avaliar(teste, len(teste), "MODELO AJUSTADO (depois do treino)")


## 13. Avaliação com logs reais

Executa a avaliação em `reais/teste_real.jsonl`, quando esse arquivo estiver disponível. Esses logs não são usados no treinamento.


In [ ]:
# Avalia os logs reais preparados na primeira etapa.
if Path("reais/teste_real.jsonl").exists():
    reais = load_dataset("json", data_files="reais/teste_real.jsonl")["train"]
    final_real = avaliar(reais, len(reais), "LOGS REAIS DO AMBIENTE")
else:
    print(
        "reais/teste_real.jsonl não encontrado. Rode preparar_reais.py "
        "com os arquivos dentro de logs_raw/."
    )


## 14. Salvar e publicar

As primeiras linhas salvam o adaptador e o tokenizer localmente. Para publicar no Hugging Face, faça login e habilite as linhas de upload.


In [ ]:
# 1. Salva o adaptador e o tokenizer no seu computador.
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adaptadores salvos localmente em ./{ADAPTER_DIR}")

In [ ]:
from pathlib import Path
import json

Path("resultados").mkdir(exist_ok=True)

with open("resultados/sintetico_modelo_v6.json", "w", encoding="utf-8") as f:
    json.dump({g: dict(v) for g, v in final.items()}, f, indent=2, ensure_ascii=False)

with open("resultados/reais_modelo_v6.json", "w", encoding="utf-8") as f:
    json.dump({g: dict(v) for g, v in final_real.items()}, f, indent=2, ensure_ascii=False)

print("Arquivos salvos em ./resultados/")


In [ ]:
from huggingface_hub import HfApi

# 2. Envia a pasta inteira para o Hugging Face
REPO_ID = HF_REPO
PASTA_LOCAL = ADAPTER_DIR

api = HfApi()
api.create_repo(repo_id=REPO_ID, private=True, exist_ok=True)

api.upload_folder(
    folder_path=PASTA_LOCAL,
    repo_id=REPO_ID,
    commit_message="Publica versão v6: fine-tuning QLoRA e validação no chaos lab",
)

print(f"Upload concluído com sucesso! Acesse: https://huggingface.co/{REPO_ID}")